# Forest diversity: spatially validated deep neural networks

Companion modeling notebook for *Geospatial Foundation Models for forest diversity mapping in structurally complex, species-rich forests*. Reads a **plot-level wide predictor table** produced by the Earth Engine / GeoTessera extraction notebook.

**Before running:** provide the CSV path and the exact column identifying the manuscript's **four management-zone blocks**.

The manuscript specifies 12 feature sets, three targets, Keras Normalization → 64 → 64 → 1, 4-zone holdout, ten seeds, and held-out permutation importance. This notebook implements these specifications. Results must be recomputed from the authorized data; numbers in the manuscript are not embedded here.

## 1. Environment and configuration

In [ ]:
# Install in a dedicated environment before starting Jupyter:
# python -m pip install numpy pandas scikit-learn tensorflow matplotlib seaborn
# Save exact versions from the successful run: python -m pip freeze > requirements-lock.txt
from pathlib import Path
import os, random, re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

DATA_PATH = Path(os.getenv("DIVERSITY_WIDE_CSV", "private/MODELING_DATASET_2023_WIDE.csv"))
BLOCK_COLUMN = os.getenv("DIVERSITY_BLOCK_COLUMN", "management_zone")
OUTPUT_DIR = Path("outputs/dnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_COLUMN = "PLOTID"
TARGETS = ("shannon_H", "species_richness", "dbh_std")
SEEDS = tuple(range(10))
EPOCHS = 100
BATCH_SIZE = 256
INNER_VALIDATION_FRACTION = 0.10
N_PERMUTATIONS = 1  # Manuscript does not specify this; increase for more stable estimates.
print('TensorFlow:', tf.__version__, 'NumPy:', np.__version__, 'pandas:', pd.__version__)

## 2. Load and validate plot-level inputs

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Provide the authorized wide-format CSV at {DATA_PATH} or set DIVERSITY_WIDE_CSV")
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()
required = [ID_COLUMN, BLOCK_COLUMN, *TARGETS]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Attach the genuine management-zone assignment before modeling.")
if df[ID_COLUMN].isna().any() or df[ID_COLUMN].duplicated().any():
    raise ValueError("Input must have one non-null row per plot ID; resolve duplicates upstream.")
if df[BLOCK_COLUMN].isna().any() or df[BLOCK_COLUMN].nunique() != 4:
    raise ValueError("The manuscript requires exactly four non-null management-zone blocks.")
if df[BLOCK_COLUMN].astype(str).eq(df[ID_COLUMN].astype(str)).all():
    raise ValueError("Management zone cannot simply be the plot identifier.")
print('Plots:', len(df), 'Zone counts:', df[BLOCK_COLUMN].value_counts().to_dict())
print('Available targets:', df[list(TARGETS)].notna().sum().to_dict())

## 3. Define the twelve predictor sets

Temporal columns must have the form `variable_YYYY-MM-DD`. Check that these groups agree with the final exported feature schema; the manuscript lists 23 bins for each temporal variable. This notebook stops when expected groups are absent rather than silently substituting other predictors.

In [ ]:
dates = sorted({m.group(1) for c in df.columns if (m := re.search(r"(\d{4}-\d{2}-\d{2})$", c))})
if len(dates) != 23:
    raise ValueError(f"Expected 23 temporal bins from manuscript Table 2; found {len(dates)}: {dates}")

# The precise exported names for the four pass-specific SAR backscatter and
# two NDI summaries must be checked against the CSV before running.
# Change these two lists to match your export if names differ; never substitute
# numeric targets, inventory attributes, coordinates, or zone identifiers.
S1_BACKSCATTER = ["VV_ASC", "VH_ASC", "VV_DESC", "VH_DESC"]
S1_NDI = ["S1_ndi", "S1_ndi_sd"]
S1_TEXTURES = ["ND_CON", "ND_DIS", "ND_ENT", "ND_HOM"]
S2_SPECTRAL = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
S2_INDICES = ["EVI", "S2_RaoQ", "EVI_RaoQ"]  # Resolve actual manuscript/export names below.
S2_TEXTURES = ["EVI_CON", "EVI_DIS", "EVI_ENT", "EVI_HOM"]

def temporal(names):
    return [f"{name}_{date}" for name in names for date in dates]

def embedding_columns(prefix):
    if prefix == 'AEF':
        cols = [f'A{i:02d}' for i in range(1, 65)]
    else:
        cols = [f'TESSERA_{i}' for i in range(128)]
    return cols

aef = embedding_columns('AEF')
tess = embedding_columns('TESSERA')
feature_sets = {
    'S1 Backscatter': temporal(S1_BACKSCATTER),
    'S1 Backscatter + NDI': temporal(S1_BACKSCATTER + S1_NDI),
    'S1 Textures': temporal(S1_TEXTURES),
    'S1 Full': temporal(S1_BACKSCATTER + S1_NDI + S1_TEXTURES),
    'S2 Spectral': temporal(S2_SPECTRAL),
    'S2 Spectral + EVI + RaoQ': temporal(S2_SPECTRAL + S2_INDICES),
    'EVI Textures': temporal(S2_TEXTURES),
    'S2 Full': temporal(S2_SPECTRAL + S2_INDICES + S2_TEXTURES),
    'All S1+S2': temporal(S1_BACKSCATTER + S1_NDI + S1_TEXTURES + S2_SPECTRAL + S2_INDICES + S2_TEXTURES),
    'AEF embeddings': aef,
    'TESSERA embeddings': tess,
    'AEF + TESSERA': aef + tess,
}
for name, cols in feature_sets.items():
    absent = [c for c in cols if c not in df.columns]
    print(f'{name}: {len(cols)} required, {len(absent)} absent; examples: {absent[:6]}')
    if absent:
        raise ValueError(f"Resolve exported names for {name} before fitting; missing {len(absent)} columns. Example: {absent[:10]}")
    if len(set(cols)) != len(cols):
        raise ValueError(f"Repeated predictor in {name}")

## 4. Four management-zone folds, ten independent seeds

For early stopping, 10% of plots from the **three training zones** are selected by a fixed seed. This is a within-training-zone validation subset and never contains the held-out test zone. The manuscript does not specify the subset-selection algorithm or random seed values; record any differences from the authors’ original run.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except RuntimeError:
        pass

def make_network(x_fit, seed):
    set_seed(seed)
    normalizer = tf.keras.layers.Normalization()
    normalizer.adapt(x_fit.astype('float32'))
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(x_fit.shape[1],)),
        normalizer,
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='mae')
    return model

def train_one(x, y, zones, held_out, seed):
    test = np.flatnonzero(zones == held_out)
    train = np.flatnonzero(zones != held_out)
    if len(test) < 2 or len(train) < 12:
        raise ValueError(f"Too few observations for held-out zone {held_out}")
    rng = np.random.default_rng(seed)
    perm = rng.permutation(train)
    n_val = max(1, round(len(train) * INNER_VALIDATION_FRACTION))
    fit_idx, val_idx = perm[n_val:], perm[:n_val]
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    x_fit = imputer.fit_transform(x.iloc[fit_idx]).astype('float32')
    x_val = imputer.transform(x.iloc[val_idx]).astype('float32')
    x_test = imputer.transform(x.iloc[test]).astype('float32')
    if not np.isfinite(x_fit).all() or not np.isfinite(x_val).all() or not np.isfinite(x_test).all():
        raise ValueError('Nonfinite predictors remain after median imputation; inspect input data.')
    model = make_network(x_fit, seed)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6),
    ]
    model.fit(x_fit, y[fit_idx], validation_data=(x_val, y[val_idx]),
              epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
    pred = model.predict(x_test, verbose=0).ravel()
    return model, x_test, test, pred

metric_rows, prediction_rows, importance_rows = [], [], []
for target in TARGETS:
    valid_target = df[target].notna() & np.isfinite(pd.to_numeric(df[target], errors='coerce'))
    table = df.loc[valid_target].reset_index(drop=True)
    y = table[target].to_numpy(dtype='float32')
    zones = table[BLOCK_COLUMN].to_numpy()
    for set_name, cols in feature_sets.items():
        x = table[cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
        for zone in sorted(pd.unique(zones), key=str):
            for seed in SEEDS:
                tf.keras.backend.clear_session()
                model, x_test, test, pred = train_one(x, y, zones, zone, seed)
                truth = y[test]
                mae = mean_absolute_error(truth, pred)
                metric_rows.append(dict(target=target, feature_set=set_name, zone=str(zone), seed=seed,
                                        n_test=len(test), n_features=len(cols), r2=r2_score(truth, pred),
                                        rmse=np.sqrt(mean_squared_error(truth, pred)), mae=mae))
                prediction_rows.extend(dict(target=target, feature_set=set_name, zone=str(zone), seed=seed,
                                            plot_id=str(table.iloc[k][ID_COLUMN]), observed=float(y[k]),
                                            predicted=float(p)) for k, p in zip(test, pred))
                # Permute held-out rows only; the same test zone is kept fixed.
                for repeat in range(N_PERMUTATIONS):
                    rng = np.random.default_rng(seed * 1009 + repeat)
                    for j, col in enumerate(cols):
                        shuffled = x_test.copy()
                        shuffled[:, j] = x_test[rng.permutation(len(test)), j]
                        shifted = model.predict(shuffled, verbose=0).ravel()
                        importance_rows.append(dict(target=target, feature_set=set_name, zone=str(zone),
                                                    seed=seed, repeat=repeat, feature=col,
                                                    delta_mae=mean_absolute_error(truth, shifted)-mae))
                print(target, set_name, zone, seed, f'R2={metric_rows[-1]["r2"]:.3f}')
                del model

metrics = pd.DataFrame(metric_rows)
predictions = pd.DataFrame(prediction_rows)
importance = pd.DataFrame(importance_rows)
summary = metrics.groupby(['target','feature_set'], as_index=False).agg(
    r2_median=('r2','median'), r2_sd=('r2','std'), rmse_median=('rmse','median'),
    rmse_sd=('rmse','std'), mae_median=('mae','median'), mae_sd=('mae','std'),
    n_folds_repeats=('r2','size'))
for name, table_out in [('metrics',metrics),('predictions',predictions),('importance',importance),('summary',summary)]:
    table_out.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
display(summary)

## 5. Held-out diagnostics and interpretation

Plots use the out-of-fold test predictions. Each plot appears ten times, once per seed. The QMD and stem-density diagnostics are optional and require these attributes in the private input table; they are never model predictors.

In [ ]:
for target in TARGETS:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, set_name in zip(axes, ('All S1+S2', 'AEF embeddings', 'TESSERA embeddings')):
        part = predictions.loc[(predictions.target == target) & (predictions.feature_set == set_name)]
        lo = min(part.observed.min(), part.predicted.min())
        hi = max(part.observed.max(), part.predicted.max())
        ax.hexbin(part.observed, part.predicted, gridsize=24, mincnt=1, bins='log', cmap='viridis')
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
        ax.set(title=set_name, xlabel='Observed', ylabel='Predicted')
    fig.suptitle(target)
    fig.savefig(OUTPUT_DIR / f'predicted_vs_observed_{target}.png', dpi=200)
    plt.close(fig)

important = importance.groupby(['target','feature_set','feature'], as_index=False).agg(
    delta_mae_mean=('delta_mae','mean'), delta_mae_sd=('delta_mae','std'))
important.to_csv(OUTPUT_DIR / 'importance_summary.csv', index=False)
for target in TARGETS:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
    for ax, set_name in zip(axes, ('All S1+S2','AEF embeddings','TESSERA embeddings')):
        top = important.loc[(important.target == target) & (important.feature_set == set_name)].nlargest(10,'delta_mae_mean').iloc[::-1]
        ax.barh(top.feature, top.delta_mae_mean, xerr=top.delta_mae_sd.fillna(0), color='steelblue')
        ax.set(title=set_name, xlabel='Held-out ΔMAE')
    fig.suptitle(target)
    fig.savefig(OUTPUT_DIR / f'permutation_importance_{target}.png', dpi=200)
    plt.close(fig)

In [ ]:
# Optional error stratification: all bins are based on the predictor-free plot attributes.
for attribute in ('QMD', 'n_trees'):
    if attribute not in df.columns:
        print('Skipping:', attribute, 'is absent')
        continue
    diagnostic = predictions.merge(df[[ID_COLUMN, attribute]].rename(columns={ID_COLUMN:'plot_id'}).assign(
        plot_id=lambda t: t.plot_id.astype(str)), on='plot_id', how='left', validate='many_to_one')
    diagnostic['absolute_error'] = (diagnostic.observed-diagnostic.predicted).abs()
    diagnostic['class'] = pd.qcut(pd.to_numeric(diagnostic[attribute],errors='coerce'), q=4,
                                  duplicates='drop')
    diagnostic.groupby(['target','feature_set','class'], observed=True).absolute_error.agg(
        ['count','median','mean']).to_csv(OUTPUT_DIR / f'error_by_{attribute}.csv')